In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Core Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
from scipy import stats
from scipy.stats import randint, uniform

## IMPORT & EXPLORE

In [3]:
import torch
from torchvision import models
from sentence_transformers import SentenceTransformer

from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as T
from sklearn.model_selection import train_test_split

In [4]:
!unzip "/content/drive/MyDrive/Data Science/flip_vlm/flip_data_vlm.zip" -d "/content/flip_data_vlm"

Streaming output truncated to the last 5000 lines.
  inflating: /content/flip_data_vlm/flip_data_vlm/flip_data_vlm/category_4/images/Хаб_Quadro_Express_1438.jpg  
  inflating: /content/flip_data_vlm/flip_data_vlm/flip_data_vlm/category_4/images/Хаб_Quadro_Infix_1542.jpg  
  inflating: /content/flip_data_vlm/flip_data_vlm/flip_data_vlm/category_4/images/Хаб_Universal_RS050_5850.jpg  
  inflating: /content/flip_data_vlm/flip_data_vlm/flip_data_vlm/category_4/images/Цветной_чехол_на_IPhone_14_Pro_Max_с_функцией_MagS_881.jpg  
  inflating: /content/flip_data_vlm/flip_data_vlm/flip_data_vlm/category_4/images/Цветной_чехол_на_IPhone_14_Pro_Max_с_функцией_MagS_882.jpg  
  inflating: /content/flip_data_vlm/flip_data_vlm/flip_data_vlm/category_4/images/Цветной_чехол_на_IPhone_14_Pro_Max_с_функцией_MagS_883.jpg  
  inflating: /content/flip_data_vlm/flip_data_vlm/flip_data_vlm/category_4/images/Цветной_чехол_на_IPhone_14_Pro_Max_с_функцией_MagS_884.jpg  
  inflating: /content/flip_data_vlm/flip_d

In [5]:
all_products = pd.read_csv("/content/flip_data_vlm/flip_data_vlm/flip_data_vlm/all_products_combined_ext.csv")
all_products = all_products.dropna()

In [6]:
all_products.sample(5)

,Unnamed: 0,title,local_image_path,product_url
1825,1825,"Жесткий диск SkyHawk Al, 10 ТБ",/root/flip/data/category_4/images/Жесткий_диск...,https://www.flip.kz/catalog?prod=2562393
11172,4556,"Худи унисекс, бордовый, единый размер",/root/flip/data/category_1/images/Худи_унисекс...,https://www.flip.kz/catalog?prod=3295164
28280,3584,Мужской Футболка Мятный,/flo_images/9e8dd4d5dd86_495dc3ce.jpg,https://www.flo.com.kz/ru/product/lumberjack-m...
3891,3891,Клавиатура Summit R1,/root/flip/data/category_4/images/Клавиатура_S...,https://www.flip.kz/catalog?prod=3348657
26547,1851,Женский Черный,/flo_images/85f4fbb35ecc_3ec6c7ab.jpg,https://www.flo.com.kz/ru/product/nine-west-af...


## CLEAN & PREPARE

In [7]:
def is_valid_image_path(path):
    path_obj = Path(path)
    return path_obj.is_file() and not path_obj.is_dir()

def convert_to_colab_path(local_path):
    if local_path.startswith("/root/flip/data/"):
        return local_path.replace("/root/flip/data/", "/content/flip_data_vlm/flip_data_vlm/flip_data_vlm/")
    elif local_path.startswith("/flo_images/"):
        return local_path.replace("/flo_images/", "/content/flip_data_vlm/flip_data_vlm/flip_data_vlm/flo_images/")
    else:
        return ""  # or local_path

# Convert local paths to Colab-compatible ones
all_products["colab_image_path"] = (
    all_products["local_image_path"]
    .fillna("")
    .astype(str)
    .apply(convert_to_colab_path)
)

# Keep only rows with valid image paths
all_products = all_products[
    all_products["colab_image_path"].apply(is_valid_image_path)
]

In [8]:
image_title_pairs = list(all_products[['colab_image_path','title']].sample(frac=1, random_state=42, replace=False).itertuples(index=False, name=None))
len(image_title_pairs)

28447

In [9]:
image_title_pairs_train, image_title_pairs_val = train_test_split(image_title_pairs, test_size=0.1, random_state=42)

In [10]:
print(len(image_title_pairs_train), len(image_title_pairs_val))

25602 2845


#### DATASET

In [11]:
class ImageTitleDataset(Dataset):

  def __init__(self, data, transform = None):

    self.data = data #(image path, text description) list of tuples
    self.transform = transform or T.Compose([
                                              T.Resize((224, 224)),
                                              T.ToTensor(),
                                              T.Normalize(mean = [0.485, 0.456, 0.406],
                                                          std = [0.229, 0.224, 0.225])
                                             ])

  def __len__(self):
    return len(self.data)

  def __getitem__(self, index):
    image_path, text = self.data[index]
    image = Image.open(image_path).convert("RGB")
    image = self.transform(image)
    return image, text



In [12]:
image_title_pairs_train_dataset = ImageTitleDataset(image_title_pairs_train)
image_title_pairs_val_dataset = ImageTitleDataset(image_title_pairs_val)

In [13]:
len(image_title_pairs_val_dataset)

2845

In [14]:
image_title_pairs_train_dataset.__getitem__(42)

(tensor([[[2.2489, 2.2489, 2.2489,  ..., 2.2489, 2.2489, 2.2489],
          [2.2489, 2.2489, 2.2489,  ..., 2.2489, 2.2489, 2.2489],
          [2.2489, 2.2489, 2.2489,  ..., 2.2489, 2.2489, 2.2489],
          ...,
          [2.2489, 2.2489, 2.2489,  ..., 2.2489, 2.2489, 2.2489],
          [2.2489, 2.2489, 2.2489,  ..., 2.2489, 2.2489, 2.2489],
          [2.2489, 2.2489, 2.2489,  ..., 2.2489, 2.2489, 2.2489]],
 
         [[2.4286, 2.4286, 2.4286,  ..., 2.4286, 2.4286, 2.4286],
          [2.4286, 2.4286, 2.4286,  ..., 2.4286, 2.4286, 2.4286],
          [2.4286, 2.4286, 2.4286,  ..., 2.4286, 2.4286, 2.4286],
          ...,
          [2.4286, 2.4286, 2.4286,  ..., 2.4286, 2.4286, 2.4286],
          [2.4286, 2.4286, 2.4286,  ..., 2.4286, 2.4286, 2.4286],
          [2.4286, 2.4286, 2.4286,  ..., 2.4286, 2.4286, 2.4286]],
 
         [[2.6400, 2.6400, 2.6400,  ..., 2.6400, 2.6400, 2.6400],
          [2.6400, 2.6400, 2.6400,  ..., 2.6400, 2.6400, 2.6400],
          [2.6400, 2.6400, 2.6400,  ...,

#### DATALOADER

In [15]:
image_title_pairs_train_dataloader = DataLoader(image_title_pairs_train_dataset, batch_size = 78, shuffle = True, drop_last = True)
image_title_pairs_val_dataloader = DataLoader(image_title_pairs_val_dataset, batch_size = 78, shuffle = True, drop_last = True)

## MODEL BUILD

In [16]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [17]:
torch.cuda.get_device_properties()

_CudaDeviceProperties(name='Tesla T4', major=7, minor=5, total_memory=15095MB, multi_processor_count=40, uuid=ec5e8a8e-1370-a989-fe0d-b8d8d6491a4b, L2_cache_size=4MB)

#### STRUCTURE

In [18]:

class FlipDualEncoderV3( torch.nn.Module ):
  def __init__(self, embedding_dimension = 256):

    super().__init__()

    # EfficientNetB3 as visual encoder
    efficientnetb3 = models.efficientnet_b3(pretrained = True)
    efficientnetb3.classifier = torch.nn.Identity()

    self.visual_encoder = efficientnetb3
    self.image_projection = torch.nn.Linear( in_features = 1536, out_features = embedding_dimension )


    # DistilLabse as text encoder
    self.text_encoder = SentenceTransformer('sentence-transformers/distiluse-base-multilingual-cased-v2')
    self.text_projection = torch.nn.Linear( in_features = 512, out_features = embedding_dimension )


  def forward(self, images, texts):
    # Image embedding
    image_encoded = self.visual_encoder(images)
    image_embedding = torch.nn.functional.normalize(
        self.image_projection(image_encoded), p=2, dim=-1
    )

    # Text embedding
    text_encoded = self.text_encoder.encode(
        texts,
        convert_to_tensor=True,
        normalize_embeddings=False
    )
    text_embedding = self.text_projection(text_encoded)
    text_embedding = torch.nn.functional.normalize(text_embedding, p=2, dim=-1)

    return image_embedding, text_embedding

In [19]:
flip_dual_encoder = FlipDualEncoderV3(embedding_dimension = 256).to(device)

/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=EfficientNet_B3_Weights.IMAGENET1K_V1`. You can also use `weights=EfficientNet_B3_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/efficientnet_b3_rwightman-b3899882.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b3_rwightman-b3899882.pth
100%|██████████| 47.2M/47.2M [00:00<00:00, 125MB/s]
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secr

modules.json:   0%|          | 0.00/341 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

2_Dense/model.safetensors:   0%|          | 0.00/1.58M [00:00<?, ?B/s]

2_Dense/pytorch_model.bin:   0%|          | 0.00/1.58M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/114 [00:00<?, ?B/s]

In [20]:
#### LOAD PRETRAINED WEIGHTS AFTER 1 TRAINING PROCCESS

flip_dual_encoder.load_state_dict( torch.load("/content/drive/MyDrive/Data Science/flip_vlm/flip_dual_encoder147m.pt", map_location = 'cuda') )

<All keys matched successfully>

In [21]:
for parameter in flip_dual_encoder.visual_encoder.parameters():
    parameter.requires_grad = True

for parameter in flip_dual_encoder.text_encoder.parameters():
    parameter.requires_grad = True

In [22]:
total_params = sum(p.numel() for p in flip_dual_encoder.parameters())
trainable_params = sum(p.numel() for p in flip_dual_encoder.parameters() if p.requires_grad)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

Total parameters: 146,348,840
Trainable parameters: 146,348,840


#### LOSS

In [23]:
def cosine_similarity_loss(image_embeddings, text_embeddings, temperature=0.07):
    """
    image_embeddings: (B, D)
    text_embeddings:  (B, D)
    """
    # Cosine similarity matrix: (B, B)
    similarity_matrix = torch.matmul(image_embeddings, text_embeddings.T) / temperature

    # Labels: diagonal is positive
    targets = torch.arange(similarity_matrix.size(0)).to(similarity_matrix.device)

    # Symmetric loss: image-to-text + text-to-image
    loss_i2t = torch.nn.functional.cross_entropy(similarity_matrix, targets)
    loss_t2i = torch.nn.functional.cross_entropy(similarity_matrix.T, targets)

    return (loss_i2t + loss_t2i) / 2

#### MODEL TRAINING

In [24]:
adam_optimizer = torch.optim.Adam( flip_dual_encoder.parameters(), lr = 1e-4 )

In [ ]:
early_stop_loss = 0.04
loss_history = []

for epoch in range(300):

  print(f"Epoch {epoch+1} started")

  #### TRAINING
  flip_dual_encoder.train()
  for step, (images, texts) in enumerate(image_title_pairs_train_dataloader):
    print(f"Train step {step}")

    images = images.to(device)

    # Flush gradients
    adam_optimizer.zero_grad()

    # Forward pass
    image_embeddings, text_embeddings = flip_dual_encoder(images, texts)

    # Compute Loss
    train_loss = cosine_similarity_loss(image_embeddings, text_embeddings)

    # Backpropagation
    train_loss.backward()

    # Update weights
    adam_optimizer.step()

    loss_history.append(train_loss.item())
    print(f"Step {step} train batch loss: {train_loss.item():.6f}")

  #### VALIDATION
  flip_dual_encoder.eval()
  val_losses = []

  with torch.no_grad():
    for val_images, val_texts in image_title_pairs_val_dataloader:
      val_images = val_images.to(device)

      val_image_embeddings, val_text_embeddings = flip_dual_encoder(val_images, val_texts)
      val_loss = cosine_similarity_loss(val_image_embeddings, val_text_embeddings)

      val_losses.append(val_loss.item())
      print(f"Val batch loss: {val_loss.item():.6f}")

  mean_val_loss = np.mean(val_losses)

  # Save and tighten threshold if loss is low enough
  if mean_val_loss <= early_stop_loss or train_loss.item() <= early_stop_loss:
    print(f"Threshold met at epoch {epoch+1} — saving model and reducing early_stop_loss")

    flip_dual_encoder_cpu = flip_dual_encoder.to('cpu')
    torch.save(flip_dual_encoder_cpu.state_dict(), "/content/drive/MyDrive/Data Science/flip_vlm/flip_dual_encoder147m.pt")
    flip_dual_encoder.to(device)

    early_stop_loss *= 0.5  # continue training with tighter threshold

  print(f"Epoch {epoch+1} train loss: {train_loss.item():.6f}")
  print(f"Epoch {epoch+1} val loss: {mean_val_loss:.6f}")


Streaming output truncated to the last 5000 lines.
Step 285 train batch loss: 0.142722
Train step 286
Step 286 train batch loss: 0.182155
Train step 287
Step 287 train batch loss: 0.198588
Train step 288
Step 288 train batch loss: 0.148477
Train step 289
Step 289 train batch loss: 0.110716
Train step 290
Step 290 train batch loss: 0.169222
Train step 291
Step 291 train batch loss: 0.172772
Train step 292
Step 292 train batch loss: 0.224213
Train step 293
Step 293 train batch loss: 0.123344
Train step 294
Step 294 train batch loss: 0.169113
Train step 295
Step 295 train batch loss: 0.125846
Train step 296
Step 296 train batch loss: 0.163804
Train step 297
Step 297 train batch loss: 0.147498
Train step 298
Step 298 train batch loss: 0.151972
Train step 299
Step 299 train batch loss: 0.180971
Train step 300
Step 300 train batch loss: 0.192575
Train step 301
Step 301 train batch loss: 0.207705
Train step 302
Step 302 train batch loss: 0.084347
Train step 303
Step 303 train batch loss: 0.12

KeyboardInterrupt: 

In [ ]:
# Early save (epoch 20)

flip_dual_encoder_cpu = flip_dual_encoder.to('cpu')
torch.save(flip_dual_encoder_cpu.state_dict(), "/content/drive/MyDrive/Data Science/flip_vlm/flip_dual_encoder147m.pt")

In [ ]:
#### CONTINUE TRAINING

early_stop_loss = 0.04
loss_history = []

for epoch in range(300):

  print(f"Epoch {epoch+1} started")

  #### TRAINING
  flip_dual_encoder.train()
  for step, (images, texts) in enumerate(image_title_pairs_train_dataloader):
    print(f"Train step {step}")

    images = images.to(device)

    # Flush gradients
    adam_optimizer.zero_grad()

    # Forward pass
    image_embeddings, text_embeddings = flip_dual_encoder(images, texts)

    # Compute Loss
    train_loss = cosine_similarity_loss(image_embeddings, text_embeddings)

    # Backpropagation
    train_loss.backward()

    # Update weights
    adam_optimizer.step()

    loss_history.append(train_loss.item())
    print(f"Step {step} train batch loss: {train_loss.item():.6f}")

  #### VALIDATION
  flip_dual_encoder.eval()
  val_losses = []

  with torch.no_grad():
    for val_images, val_texts in image_title_pairs_val_dataloader:
      val_images = val_images.to(device)

      val_image_embeddings, val_text_embeddings = flip_dual_encoder(val_images, val_texts)
      val_loss = cosine_similarity_loss(val_image_embeddings, val_text_embeddings)

      val_losses.append(val_loss.item())
      print(f"Val batch loss: {val_loss.item():.6f}")

  mean_val_loss = np.mean(val_losses)

  # Save and tighten threshold if loss is low enough
  if mean_val_loss <= early_stop_loss or train_loss.item() <= early_stop_loss:
    print(f"Threshold met at epoch {epoch+1} — saving model and reducing early_stop_loss")

    flip_dual_encoder_cpu = flip_dual_encoder.to('cpu')
    torch.save(flip_dual_encoder_cpu.state_dict(), "/content/drive/MyDrive/Data Science/flip_vlm/flip_dual_encoder147m.pt")
    flip_dual_encoder.to(device)

    early_stop_loss *= 0.5  # continue training with tighter threshold

  print(f"Epoch {epoch+1} train loss: {train_loss.item():.6f}")
  print(f"Epoch {epoch+1} val loss: {mean_val_loss:.6f}")


Streaming output truncated to the last 5000 lines.
Step 277 train batch loss: 0.021738
Train step 278
Step 278 train batch loss: 0.043900
Train step 279
Step 279 train batch loss: 0.037582
Train step 280
Step 280 train batch loss: 0.103610
Train step 281
Step 281 train batch loss: 0.053432
Train step 282
Step 282 train batch loss: 0.027630
Train step 283
Step 283 train batch loss: 0.088997
Train step 284
Step 284 train batch loss: 0.113932
Train step 285
Step 285 train batch loss: 0.032285
Train step 286
Step 286 train batch loss: 0.032717
Train step 287
Step 287 train batch loss: 0.065308
Train step 288
Step 288 train batch loss: 0.097488
Train step 289
Step 289 train batch loss: 0.101736
Train step 290
Step 290 train batch loss: 0.032820
Train step 291
Step 291 train batch loss: 0.082408
Train step 292
Step 292 train batch loss: 0.044373
Train step 293
Step 293 train batch loss: 0.059125
Train step 294
Step 294 train batch loss: 0.029920
Train step 295
Step 295 train batch loss: 0.09